In [25]:
import polars as pl
from polars import col

In [ ]:
trade_schema = pl.Struct([
    pl.Field("e", pl.String),
    pl.Field("E", pl.Int64),
    pl.Field("s", pl.String),
    pl.Field("t", pl.Int64),
    pl.Field("p", pl.String), # Must stay String here to handle quotes
    pl.Field("q", pl.String), # Must stay String here to handle quotes
    pl.Field("T", pl.Int64),
    pl.Field("m", pl.Boolean),
    pl.Field("M", pl.Boolean)
])

df_trades = (
    pl.read_csv(
        "/Users/oceansxyz/alphanode-metal/logs/ingest.jsonl",
        has_header=False,
        new_columns=["raw_line"],
        separator="\n"
    )
    .filter(pl.col("raw_line").str.contains('"e":"trade"'))
    .select([
        # 1. Cast your log timestamp to integer right away
        pl.col("raw_line").str.extract(r"^(\d+)").cast(pl.Int64).alias("log_timestamp"),
        
        # 2. Extract and decode the raw JSON object
        pl.col("raw_line").str.extract(r"(\{.*\})$").str.json_decode(dtype=trade_schema).alias("data")
    ])
    .select([
        pl.col("log_timestamp"),
        # 3. Pull fields and cast them to proper types immediately
        pl.col("data").struct.field("e").alias("event_type"),
        pl.col("data").struct.field("E").alias("event_time"),
        pl.col("data").struct.field("s").alias("symbol"),
        pl.col("data").struct.field("t").alias("trade_id"),
        pl.col("data").struct.field("p").cast(pl.Float64).alias("price"),    # Cast early
        pl.col("data").struct.field("q").cast(pl.Float64).alias("quantity"), # Cast early
        pl.col("data").struct.field("T").alias("trade_time"),
        pl.col("data").struct.field("m").alias("is_buyer_maker")
    ])
)

#1780678107499 {"e":"trade","E":1780678107512,"s":"BTCUSDT","t":233870993,"p":"60982.01000000","q":"0.00492000","T":1780678107512,"m":false,"M":true}

In [21]:
df_trades.head()

log_timestamp,event_type,event_time,symbol,trade_id,price,quantity,trade_time,is_buyer_maker
i64,str,i64,str,i64,f64,f64,i64,bool
1780678107499,"""trade""",1780678107512,"""BTCUSDT""",233870993,60982.01,0.00492,1780678107512,false
1780678107549,"""trade""",1780678107563,"""USDCUSDT""",119541896,1.00035,1082.0,1780678107563,true
1780678107600,"""trade""",1780678107614,"""USDCUSDT""",119541897,1.00035,311.0,1780678107613,true
1780678107600,"""trade""",1780678107614,"""USDCUSDT""",119541898,1.00035,61.0,1780678107613,true
1780678107621,"""trade""",1780678107635,"""USDCUSDT""",119541899,1.00035,9.0,1780678107634,true


In [33]:
df_trades.filter(col("symbol")=="BTCUSDT")\
    .select(col("log_timestamp"), col("event_time"), col("symbol"), col("price"), col("quantity"), col("is_buyer_maker"))

log_timestamp,event_time,symbol,price,quantity,is_buyer_maker
i64,i64,str,f64,f64,bool
1780678107499,1780678107512,"""BTCUSDT""",60982.01,0.00492,false
1780678107702,1780678107715,"""BTCUSDT""",60982.01,0.00019,false
1780678107975,1780678107988,"""BTCUSDT""",60982.01,0.00048,false
1780678108122,1780678108136,"""BTCUSDT""",60974.01,0.00032,false
1780678108297,1780678108310,"""BTCUSDT""",60974.01,0.00081,false
…,…,…,…,…,…
1780803075021,1780803075104,"""BTCUSDT""",61636.08,0.01,false
1780803075139,1780803075227,"""BTCUSDT""",61636.08,0.00759,false
1780803076016,1780803076103,"""BTCUSDT""",61640.0,0.0093,false


In [34]:
df_trades.columns

['log_timestamp',
 'event_type',
 'event_time',
 'symbol',
 'trade_id',
 'price',
 'quantity',
 'trade_time',
 'is_buyer_maker']

In [24]:
df_trades.filter(pl.col("event_type")!="trade")

log_timestamp,event_type,event_time,symbol,trade_id,price,quantity,trade_time,is_buyer_maker
i64,str,i64,str,i64,f64,f64,i64,bool


In [ ]:
#1780678108431 {"lastUpdateId":1391613837,

# "bids":[["60951.74000000","0.56900000"],["60948.99000000","0.02299000"],["60948.98000000","0.07702000"],
# ["60948.65000000","0.02717000"],["60948.64000000","0.02207000"]],

# "asks":[["60951.75000000","0.25156000"],["60955.52000000","0.00019000"],["60958.27000000","0.06895000"],
# ["60958.68000000","0.00013000"],["60960.35000000","0.16100000"]]}


In [ ]:
# Define the order book schema
orderbook_schema = pl.Struct([
    pl.Field("lastUpdateId", pl.Int64),
    pl.Field("bids", pl.List(pl.List(pl.String))), # Array of [price, qty] pairs
    pl.Field("asks", pl.List(pl.List(pl.String)))  # Array of [price, qty] pairs
])

# Read and process ONLY lines containing order book updates
df_orderbooks = (
    pl.read_csv("/Users/oceansxyz/alphanode-metal/logs/ingest.jsonl", has_header=False, new_columns=["raw_line"], separator="\n")
    .filter(pl.col("raw_line").str.contains('"lastUpdateId"'))  # Filter step
    .select([
        pl.col("raw_line").str.extract(r"^(\d+)").alias("log_timestamp"),
        pl.col("raw_line").str.extract(r"(\{.*\})$").str.json_decode(dtype=orderbook_schema).alias("data")
    ])
    .unnest("data")
)



In [18]:
df_orderbooks

log_timestamp,lastUpdateId,bids,asks
str,i64,list[list[str]],list[list[str]]
"""1780678108006""",4487691895,"[[""60982.00000000"", ""0.19394000""], [""60981.99000000"", ""0.00081000""], … [""60980.08000000"", ""0.00009000""]]","[[""60982.01000000"", ""2.05686000""], [""60982.02000000"", ""0.00018000""], … [""60984.00000000"", ""0.00009000""]]"
"""1780678108431""",1391613837,"[[""60951.74000000"", ""0.56900000""], [""60948.99000000"", ""0.02299000""], … [""60948.64000000"", ""0.02207000""]]","[[""60951.75000000"", ""0.25156000""], [""60955.52000000"", ""0.00019000""], … [""60960.35000000"", ""0.16100000""]]"
"""1780678108431""",189452875,"[[""1.00035000"", ""3257506.00000000""], [""1.00034000"", ""1273771.00000000""], … [""1.00031000"", ""4310845.00000000""]]","[[""1.00036000"", ""4337747.00000000""], [""1.00037000"", ""7850.00000000""], … [""1.00040000"", ""5643273.00000000""]]"
"""1780678109006""",4487692484,"[[""60974.00000000"", ""1.24715000""], [""60973.99000000"", ""0.00018000""], … [""60972.82000000"", ""0.06338000""]]","[[""60974.01000000"", ""0.06428000""], [""60974.02000000"", ""0.00090000""], … [""60976.01000000"", ""0.00009000""]]"
"""1780678109420""",1391614062,"[[""60928.97000000"", ""0.09805000""], [""60928.76000000"", ""0.04606000""], … [""60927.80000000"", ""0.00010000""]]","[[""60931.82000000"", ""0.21444000""], [""60934.16000000"", ""1.01561000""], … [""60936.97000000"", ""0.02058000""]]"
…,…,…,…
"""1780803076350""",1402135787,"[[""61633.22000000"", ""0.98617000""], [""61630.37000000"", ""0.04100000""], … [""61630.13000000"", ""0.04654000""]]","[[""61633.23000000"", ""0.15297000""], [""61635.40000000"", ""0.00018000""], … [""61636.99000000"", ""0.02436000""]]"
"""1780803076350""",190933395,"[[""1.00016000"", ""604311.00000000""], [""1.00015000"", ""3102045.00000000""], … [""1.00012000"", ""3000213.00000000""]]","[[""1.00017000"", ""1924573.00000000""], [""1.00018000"", ""2141582.00000000""], … [""1.00021000"", ""4555912.00000000""]]"
"""1780803076927""",4522006591,"[[""61639.99000000"", ""2.04318000""], [""61639.98000000"", ""0.00045000""], … [""61639.88000000"", ""0.05658000""]]","[[""61640.00000000"", ""0.01368000""], [""61640.01000000"", ""0.00081000""], … [""61641.00000000"", ""0.00033000""]]"


In [36]:
zst_path ="/Users/oceansxyz/alphanode-metal/logs/ingest/ingest_20260608_03.jsonl.zst"

In [37]:
df_zst = pl.read_ndjson(zst_path)

df_zst.head()

recv_ts_utc_ms,symbol,kind,ts_ms,trade_px,trade_qty
i64,str,str,i64,f64,f64
1780890563237,"""USDCUSDT""","""trade""",1780890563207,1.00022,670.0
1780890563239,"""USDCUSDT""","""trade""",1780890563207,1.00022,1341.0
1780890563613,"""USDCUSDT""","""trade""",1780890563581,1.00022,134.0
1780890563643,"""BTCUSDT""","""trade""",1780890563612,63109.99,0.00569
1780890563844,"""USDCUSDT""","""trade""",1780890563814,1.00022,23.0


In [38]:
df_zst.write_parquet("/Users/oceansxyz/alphanode-metal/logs/ingest/ingest_20260608_08.parquet", compression="zstd")